In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
from langchain_core.prompts import ChatPromptTemplate

from app.vectordb import load_vector_db
from app.retriever import get_retriever

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
vector_db = load_vector_db()

retriever = get_retriever(vector_db)

In [18]:
question = "Tell me medical information about diabetes."

docs = retriever.invoke(question)

In [19]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [20]:
print(context[:1000])

patient_id: P033
name: Timothy Walker
age: 60
gender: Male
blood_group: AB+
weight_kg: 80
height_cm: 169
medical_history: Type 2 Diabetes since 2012; Neuropathy; Retinopathy; Hypertension
current_diseases: Type 2 Diabetes; Hypertension; Diabetic Neuropathy; Retinopathy; Cold with Cough
current_medications: Insulin Degludec 30 units daily; Lisinopril 20mg; Pregabalin 150mg twice daily; Aspirin 81mg
allergies: Sulfonylureas; Insulin analogs
last_visit_date: 2026-06-14
primary_physician: Dr. Michael Chen
emergency_contact: 4765432108

patient_id: P003
name: Robert Williams
age: 72
gender: Male
blood_group: B+
weight_kg: 75
height_cm: 170
medical_history: Type 2 Diabetes since 2008; CKD Stage 2 since 2019; Hyperlipidemia; MI 2015
current_diseases: Type 2 Diabetes; Chronic Kidney Disease; Hyperlipidemia; Fever and Cold
current_medications: Insulin Glargine 20 units nightly; Atorvastatin 20mg daily; Furosemide 40mg daily
allergies: None
last_visit_date: 2026-06-18
primary_physician: Dr. Sara

In [21]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not found in the context, say:

"I couldn't find that information in the uploaded documents."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [22]:
formatted_prompt = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

In [23]:
from langchain_ollama import ChatOllama

In [24]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [25]:
response = llm.invoke(formatted_prompt)

print(response.content)

 The provided context includes information about three patients who have Type 2 Diabetes. Here's a summary of their current medications and medical history related to diabetes:

1. Timothy Walker (P033) - Insulin Degludec 30 units daily, last visit date: 2026-06-14
2. Robert Williams (P003) - Insulin Glargine 20 units nightly, last visit date: 2026-06-18
3. Margaret Scott (P028) - Semaglutide 1mg weekly, last visit date: 2026-06-20

In addition, the context also provides a risk assessment for cold medications for patients with diabetes, which is considered moderate. Safe options include Acetaminophen, Guaifenesin, Dextromethorphan, Loratadine/Cetirizine/Fexofenadine, nasal saline spray, and Oxymetazoline nasal. Pseudoephedrine and Phenylephrine should be used with caution as they may cause mild hyperglycemia, while oral corticosteroids (Prednisone, Dexamethasone) are contraindicated/avoided due to significant hyperglycemia.


In [26]:
system_prompt = """
You are an AI assistant for question answering.

Rules:

1. Answer ONLY from the provided context.

2. Never make up information.

3. If the answer is unavailable, reply:

"I couldn't find that information in the uploaded documents."

4. Keep answers concise.

5. If multiple context chunks are available, combine them logically.
"""

In [27]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),

        ("human",
"""
Context:

{context}

Question:

{question}
""")
    ]
)

In [28]:
context = "\n\n".join(
    f"Source: {doc.metadata.get('source')}, "
    f"Page: {doc.metadata.get('page', 'N/A')}\n"
    f"{doc.page_content}"
    for doc in docs
)

In [29]:
from app.prompts import RAG_PROMPT

In [30]:
formatted = RAG_PROMPT.invoke(
    {
        "context": context,
        "question": question
    }
)

In [31]:
response = llm.invoke(formatted)

print(response.content)

 The provided context includes three patient records, one of which (P033) has Type 2 Diabetes listed as a current disease. Additionally, there is a section from a medical handbook (Page 2, Section 2.2) that discusses safe cold medications for patients with diabetes, specifically mentioning moderate risk assessment and safe options like Acetaminophen, Guaifenesin, Dextromethorphan, Loratadine/Cetirizine/Fexofenadine, Nasal saline spray, Oxymetazoline nasal, and Pseudoephedrine with caution. However, the context does not provide detailed information about diabetes itself.

For more comprehensive information about diabetes, I would recommend consulting a reliable medical resource such as the Mayo Clinic or the American Diabetes Association.
